# CS 212 — Lab 5: TensorFlow with Keras
## KMNIST Image Classification

**Maksym Kholodenko · 10/25/2026 · Group B**

**Objective:** Classification of 10 classical Japanese Kuzushiji characters using TensorFlow and Keras. KMNIST contains 60,000 training images and 10,000 test images, each 28 × 28 pixels in grayscale.

**References:** [TensorFlow basic image classification tutorial](https://www.tensorflow.org/tutorials/keras/classification) · [KMNIST dataset documentation](https://www.tensorflow.org/datasets/catalog/kmnist)


## Step 1 — Import Libraries

TensorFlow, TensorFlow Datasets, NumPy, and Matplotlib. Colab dependencies: `tensorflow-datasets` and `importlib_resources`.


In [ ]:
# Dataset dependencies for Google Colab.
%pip -q install tensorflow-datasets importlib_resources


In [3]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


## Step 2 — Load the KMNIST Dataset

KMNIST training and test splits converted to NumPy arrays.


In [5]:
# Supervised KMNIST training and test splits.
(ds_train, ds_test), ds_info = tfds.load(
    'kmnist',
    split=['train', 'test'],
    shuffle_files=True,
    as_supervised=True,
    with_info=True,
)

# TensorFlow tensors converted to NumPy arrays.
def convert_to_numpy(dataset):
    images = []
    labels = []
    for img, lbl in dataset:
        images.append(img.numpy())
        labels.append(lbl.numpy())
    return np.array(images), np.array(labels)

print("Converting training data to NumPy...")
train_images, train_labels = convert_to_numpy(ds_train)
print("Converting test data to NumPy...")
test_images, test_labels = convert_to_numpy(ds_test)

# Class names in KMNIST label order.
class_names = ['o', 'ki', 'su', 'tsu', 'na', 'ha', 'ma', 'ya', 're', 'wo']

print("KMNIST loaded successfully.")

ModuleNotFoundError: No module named 'importlib_resources'

## Step 3 — Explore the Data

Image dimensions, dataset sizes, and class labels.


In [ ]:
print("Training images shape:", train_images.shape)
print("Training labels:", len(train_labels))
print("Test images shape:", test_images.shape)
print("Test labels:", len(test_labels))
print("Number of classes:", ds_info.features['label'].num_classes)
print("First 10 training labels:", train_labels[:10])

assert train_images.shape == (60000, 28, 28, 1)
assert test_images.shape == (10000, 28, 28, 1)
assert len(train_labels) == 60000
assert len(test_labels) == 10000

In [ ]:
# Original grayscale image and pixel scale.
plt.figure(figsize=(5, 4))
plt.imshow(train_images[0].squeeze(), cmap='gray')
plt.colorbar()
plt.title(f"Original image: {class_names[int(train_labels[0])]}")
plt.grid(False)
plt.show()

## Step 4 — Preprocess the Data

Pixel values normalized from 0–255 to 0–1.


In [ ]:
# Pixel normalization for training and test images.
train_images = train_images / 255.0
test_images = test_images / 255.0

print("Training pixel range:", train_images.min(), "to", train_images.max())
print("Test pixel range:", test_images.min(), "to", test_images.max())

In [ ]:
# First 25 normalized training images with class labels.
plt.figure(figsize=(10, 10))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(train_images[i].squeeze(), cmap=plt.cm.binary)
    plt.xlabel(class_names[int(train_labels[i])])
plt.tight_layout()
plt.show()

## Step 5 — Build the Model

Architecture: Flatten → Dense(128, ReLU) → Dense(10, logits).


In [ ]:
# Baseline architecture from the Fashion MNIST tutorial.
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28, 1)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(10),
])

model.summary()

## Step 6 — Compile the Model

Optimizer: Adam. Loss: sparse categorical cross-entropy. Metric: accuracy.


In [ ]:
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy'],
)

## Step 7 — Train the Model

Training duration: 10 epochs.


In [ ]:
# Per-epoch training metrics for the baseline and Dropout comparison.
history = model.fit(train_images, train_labels, epochs=10)

## Step 8 — Evaluate the Model

Model accuracy and loss on the test dataset.


In [ ]:
test_loss, test_accuracy = model.evaluate(test_images, test_labels, verbose=2)
baseline_train_accuracy = history.history['accuracy'][-1]

print(f"Baseline final training accuracy: {baseline_train_accuracy:.4f}")
print(f"Baseline test accuracy: {test_accuracy:.4f}")
print(f"Baseline test loss: {test_loss:.4f}")

## Step 9 — Make Predictions

Softmax class probabilities and prediction visualizations.


In [ ]:
# Softmax conversion of logits to class probabilities.
probability_model = tf.keras.Sequential([
    model,
    tf.keras.layers.Softmax(),
])
predictions = probability_model.predict(test_images, verbose=0)

print("First prediction probabilities:", predictions[0])
print("Predicted label:", class_names[int(np.argmax(predictions[0]))])
print("True label:", class_names[int(test_labels[0])])

In [ ]:
# Image, predicted class, and class probability distribution.
def plot_image(i, predictions_array, true_labels, images):
    true_label = int(true_labels[i])
    image = images[i].squeeze()
    predicted_label = int(np.argmax(predictions_array))

    plt.grid(False)
    plt.xticks([])
    plt.yticks([])
    plt.imshow(image, cmap=plt.cm.binary)

    label_color = 'blue' if predicted_label == true_label else 'red'
    plt.xlabel(
        f"{class_names[predicted_label]} {100 * np.max(predictions_array):.0f}% "
        f"({class_names[true_label]})",
        color=label_color,
    )


def plot_value_array(i, predictions_array, true_labels):
    true_label = int(true_labels[i])
    predicted_label = int(np.argmax(predictions_array))

    plt.grid(False)
    plt.xticks(range(10))
    plt.yticks([])
    bars = plt.bar(range(10), predictions_array, color='#777777')
    plt.ylim([0, 1])
    bars[predicted_label].set_color('red')
    bars[true_label].set_color('blue')

In [ ]:
# First test image and its ten class probabilities.
i = 0
plt.figure(figsize=(9, 3))
plt.subplot(1, 2, 1)
plot_image(i, predictions[i], test_labels, test_images)
plt.subplot(1, 2, 2)
plot_value_array(i, predictions[i], test_labels)
plt.xticks(range(10), class_names, rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Fifteen predictions: blue = correct, red = incorrect.
num_rows = 5
num_cols = 3
num_images = num_rows * num_cols
plt.figure(figsize=(2 * 2 * num_cols, 2 * num_rows))
for i in range(num_images):
    plt.subplot(num_rows, 2 * num_cols, 2 * i + 1)
    plot_image(i, predictions[i], test_labels, test_images)
    plt.subplot(num_rows, 2 * num_cols, 2 * i + 2)
    plot_value_array(i, predictions[i], test_labels)
plt.tight_layout()
plt.show()

In [ ]:
# Single-image prediction with an added batch dimension.
img = test_images[1]
print("Single image shape:", img.shape)
img = np.expand_dims(img, axis=0)
print("Single image batch shape:", img.shape)

predictions_single = probability_model.predict(img, verbose=0)
predicted_index = int(np.argmax(predictions_single[0]))
print("Predicted class:", class_names[predicted_index])
print("Actual class:", class_names[int(test_labels[1])])

plt.figure(figsize=(7, 3))
plot_value_array(1, predictions_single[0], test_labels)
plt.xticks(range(10), class_names, rotation=45)
plt.tight_layout()
plt.show()

## Optional Bonus — Dropout and Overfitting


In [ ]:
# Second model with 20% Dropout after the hidden layer.
dropout_model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28, 1)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(10),
])

dropout_model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy'],
)

dropout_history = dropout_model.fit(train_images, train_labels, epochs=10)

In [ ]:
# Final training and test metrics for both model architectures.
dropout_test_loss, dropout_test_accuracy = dropout_model.evaluate(
    test_images, test_labels, verbose=2
)
dropout_train_accuracy = dropout_history.history['accuracy'][-1]

baseline_gap = baseline_train_accuracy - test_accuracy
dropout_gap = dropout_train_accuracy - dropout_test_accuracy

print("Baseline:")
print(f"  Final training accuracy: {baseline_train_accuracy:.4f}")
print(f"  Test accuracy:           {test_accuracy:.4f}")
print(f"  Train/test accuracy gap: {baseline_gap:.4f}")

print("Dropout (0.2):")
print(f"  Final training accuracy: {dropout_train_accuracy:.4f}")
print(f"  Test accuracy:           {dropout_test_accuracy:.4f}")
print(f"  Train/test accuracy gap: {dropout_gap:.4f}")

print(f"Test accuracy change (Dropout - baseline): {dropout_test_accuracy - test_accuracy:+.4f}")
print(f"Gap change (Dropout - baseline): {dropout_gap - baseline_gap:+.4f}")

if dropout_test_accuracy > test_accuracy:
    print("Dropout improved test accuracy in this run.")
elif dropout_test_accuracy < test_accuracy:
    print("Dropout reduced test accuracy in this run.")
else:
    print("Both models achieved the same test accuracy in this run.")

if dropout_gap < baseline_gap:
    print("The Dropout model has a smaller training/test accuracy gap.")
else:
    print("The Dropout model did not reduce the training/test accuracy gap in this run.")